**Authenticate and Download the Dataset**

In [13]:
# 1. Setup Kaggle authentication using your generated token
from google.colab import userdata
import os


kaggle_token = userdata.get('KAGGLE_TOKEN')

!mkdir -p ~/.kaggle
!echo "{kaggle_token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

# 2. Download and extract the dataset
!kaggle datasets download -d rohanrao/sudoku
!unzip -q sudoku.zip

print("Dataset downloaded and extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/rohanrao/sudoku
License(s): CC0-1.0
sudoku.zip: Skipping, found more recently modified local copy (use --force to force download)
replace sudoku.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Dataset downloaded and extracted successfully!


**Load the Data and Convert a Puzzle String to a 9x9 Grid**

In [14]:
import pandas as pd
df = pd.read_csv('sudoku.csv', nrows=5)
print("Actual column names:", df.columns.tolist())

# Function to convert an 81-character string into a 9x9 2D list
def string_to_board(s):
    return [[int(char) for char in s[i:i+9]] for i in range(0, 81, 9)]

# Function to print the board neatly
def print_board(board):
    for row in board:
        print(" ".join(str(c) if c != 0 else '.' for c in row))

# Grab the first puzzle string from the dataset
sample_quiz_str = df.iloc[0]['puzzle']
board = string_to_board(sample_quiz_str)

print("Initial Sudoku Puzzle:")
print_board(board)

Actual column names: ['puzzle', 'solution']
Initial Sudoku Puzzle:
. 7 . . . . . 4 3
. 4 . . . 9 6 1 .
8 . . 6 3 4 9 . .
. 9 4 . 5 2 . . .
3 5 8 4 6 . . 2 .
. . . 8 . . 5 3 .
. 8 . . 7 . . 9 1
9 . 2 1 . . . . 5
. . 7 . 4 . 8 . 2


**Backtracking**

In [15]:
def is_valid(board, row, col, num):
    # Check row and column
    for i in range(9):
        if board[row][i] == num or board[i][col] == num:
            return False

    # Check 3x3 subgrid
    start_row, start_col = 3 * (row // 3), 3 * (col // 3)
    for i in range(3):
        for j in range(3):
            if board[start_row + i][start_col + j] == num:
                return False
    return True

def solve_backtracking(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                for num in range(1, 10):
                    if is_valid(board, r, c, num):
                        board[r][c] = num
                        if solve_backtracking(board):
                            return True
                        board[r][c] = 0
                return False
    return True

# Test Backtracking on our sample puzzle
test_board = string_to_board(sample_quiz_str)
solve_backtracking(test_board)
print("\nSolved using Backtracking:")
print_board(test_board)


Solved using Backtracking:
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**[A] search**

In [16]:
import heapq

def get_empty_cell(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                return r, c
    return None

def heuristic(board):
    # Heuristic: Number of remaining empty cells
    return sum(row.count(0) for row in board)

def solve_a_star(initial_board):
    # Priority queue stores tuples of: (f_score, g_score, board)
    pq = []
    g_score = 0
    h_score = heuristic(initial_board)
    heapq.heappush(pq, (g_score + h_score, g_score, initial_board))

    while pq:
        f, g, current = heapq.heappop(pq)

        empty = get_empty_cell(current)
        if not empty:
            return current  # Goal state found!

        r, c = empty
        for num in range(1, 10):
            if is_valid(current, r, c, num):
                # Create a deep copy of the board
                new_board = [row[:] for row in current]
                new_board[r][c] = num

                new_g = g + 1
                new_f = new_g + heuristic(new_board)
                heapq.heappush(pq, (new_f, new_g, new_board))

    return None

# Test A* Search on our sample puzzle
a_star_board = string_to_board(sample_quiz_str)
solved_a_star = solve_a_star(a_star_board)

print("\nSolved using A* Search:")
if solved_a_star:
    print_board(solved_a_star)
else:
    print("No solution found.")


Solved using A* Search:
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**Hill Climbing**

In [17]:
import random
import time

def _init_box_fill(initial):
    board = [row[:] for row in initial]
    free = []
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            cells = [(br+i, bc+j) for i in range(3) for j in range(3)]
            present = {board[r][c] for r, c in cells if board[r][c] != 0}
            missing = [n for n in range(1, 10) if n not in present]
            random.shuffle(missing)
            empties = [(r, c) for r, c in cells if board[r][c] == 0]
            for (r, c), v in zip(empties, missing):
                board[r][c] = v
            free.append(empties)
    return board, free

def _row_cost(b, r):
    return 9 - len(set(b[r]))

def _col_cost(b, c):
    return 9 - len({b[i][c] for i in range(9)})

def _local(b, rows, cols):
    return sum(_row_cost(b, r) for r in rows) + sum(_col_cost(b, c) for c in cols)

def total_conflicts(b):
    return sum(_row_cost(b, i) + _col_cost(b, i) for i in range(9))

def solve_hill_climbing(initial_board, max_restarts=200, max_steps=300,
                        max_sideways=30, time_limit=10, seed=None):
    if seed is not None:
        random.seed(seed)
    t0 = time.perf_counter()
    stats = {'nodes': 0, 'restarts': 0, 'final_conflicts': None}
    best_overall = None

    for _ in range(max_restarts):
        stats['restarts'] += 1
        board, free = _init_box_fill(initial_board)   # boxes always valid
        cost = total_conflicts(board)
        sideways = 0

        for _ in range(max_steps):
            if cost == 0:
                stats['final_conflicts'] = 0
                return board, stats
            if time.perf_counter() - t0 > time_limit:
                return None, stats

            best_delta, best_moves = None, []
            for cells in free:                        # neighbors: swap 2 free cells in a box
                for a in range(len(cells)):
                    for b in range(a+1, len(cells)):
                        (r1, c1), (r2, c2) = cells[a], cells[b]
                        if board[r1][c1] == board[r2][c2]:
                            continue
                        rows, cols = {r1, r2}, {c1, c2}
                        before = _local(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        after = _local(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        stats['nodes'] += 1
                        d = after - before
                        if best_delta is None or d < best_delta:
                            best_delta, best_moves = d, [(r1, c1, r2, c2)]
                        elif d == best_delta:
                            best_moves.append((r1, c1, r2, c2))

            if best_delta is None or best_delta > 0:
                break                                 # local minimum -> restart
            if best_delta == 0:
                sideways += 1
                if sideways > max_sideways:
                    break
            else:
                sideways = 0

            r1, c1, r2, c2 = random.choice(best_moves)
            board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
            cost += best_delta

        best_overall = cost if best_overall is None else min(best_overall, cost)
        stats['final_conflicts'] = best_overall

    return None, stats

# --- Run the algorithm using your correct sample variable ---
res, st = solve_hill_climbing(string_to_board(sample_quiz_str))
print("\nHill Climbing Stats:", st)
if res:
    print("\nSolved using Hill Climbing:")
    print_board(res)
else:
    print("\nNot solved within limits (Hit local minimum or timeout).")


Hill Climbing Stats: {'nodes': 2047, 'restarts': 1, 'final_conflicts': 0}

Solved using Hill Climbing:
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**Benchmark and comparison**

In [18]:
import time
import pandas as pd

# 1. Helper function to turn board back into a flat string for verification
def board_to_string(board):
    return "".join(str(cell) for row in board for cell in row)

# 2. Tracked Backtracking Solver (returns board and stats)
def solve_backtracking_tracked(initial_board):
    board = [row[:] for row in initial_board]
    stats = {'nodes': 0}

    def backtrack(b):
        stats['nodes'] += 1
        for r in range(9):
            for c in range(9):
                if b[r][c] == 0:
                    for num in range(1, 10):
                        if is_valid(b, r, c, num):
                            b[r][c] = num
                            if backtrack(b):
                                return True
                            b[r][c] = 0
                    return False
        return True

    success = backtrack(board)
    return (board if success else None), stats

# 3. Tracked A* Solver (returns board and stats)
def solve_a_star_tracked(initial_board):
    board = [row[:] for row in initial_board]
    import heapq

    def get_empty_cell(b):
        for r in range(9):
            for c in range(9):
                if b[r][c] == 0:
                    return r, c
        return None

    def heuristic(b):
        return sum(row.count(0) for row in b)

    pq = []
    g_score = 0
    h_score = heuristic(board)
    heapq.heappush(pq, (g_score + h_score, g_score, board))
    nodes = 0

    while pq:
        f, g, current = heapq.heappop(pq)
        nodes += 1

        empty = get_empty_cell(current)
        if not empty:
            return current, {'nodes': nodes}

        r, c = empty
        for num in range(1, 10):
            if is_valid(current, r, c, num):
                new_board = [row[:] for row in current]
                new_board[r][c] = num
                new_g = g + 1
                heapq.heappush(pq, (new_g + heuristic(new_board), new_g, new_board))

    return None, {'nodes': nodes}

# 4. Dictionary mapping algorithm names to their execution functions
algorithms = {
    'Backtracking': solve_backtracking_tracked,
    'A*': solve_a_star_tracked,
    'Hill Climbing': solve_hill_climbing,  # Uses your optimized hill climbing from earlier
}

# 5. Run Benchmark across your DataFrame rows (e.g., first 20 rows)
records = []
df_subset = df.head(20)

for i, row in df_subset.iterrows():
    puzzle_board = string_to_board(row['puzzle'])
    for name, fn in algorithms.items():
        t = time.perf_counter()
        result, stats = fn(puzzle_board)
        dt = time.perf_counter() - t

        solved = result is not None
        correct = solved and board_to_string(result) == row['solution']

        records.append({
            'puzzle': i,
            'algorithm': name,
            'solved': solved,
            'correct': correct,
            'time_s': round(dt, 4),
            'nodes': stats.get('nodes', 0)
        })

results = pd.DataFrame(records)
print(results.to_string(index=False))

# 6. Generate and Print Summary Table
summary = results.groupby('algorithm').agg(
    success_rate=('correct', 'mean'),
    avg_time_s=('time_s', 'mean'),
    avg_nodes=('nodes', 'mean'),
).round(4)

print("\nSummary:\n", summary)

 puzzle     algorithm  solved  correct  time_s  nodes
      0  Backtracking    True     True  0.0037    146
      0            A*    True     True  0.0056    192
      0 Hill Climbing    True     True  0.0742   4272
      1  Backtracking    True     True  0.0073    215
      1            A*    True     True  0.0277    227
      1 Hill Climbing    True     True  0.0524   2430
      2  Backtracking    True     True  0.0007     58
      2            A*    True     True  0.0022     98
      2 Hill Climbing    True     True  0.0251   1850
      3  Backtracking    True     True  0.0018    127
      3            A*    True     True  0.0041    191
      3 Hill Climbing    True     True  0.1257   9858
      4  Backtracking    True     True  0.0022    154
      4            A*    True     True  0.0039    181
      4 Hill Climbing    True     True  0.0451   3496

Summary:
                success_rate  avg_time_s  avg_nodes
algorithm                                         
A*                     